# 08. Access, deprivation and vehicle availability

This notebook relates walking distance to food to the social conditions of the
places it was measured in, and carries the block level measurements up to the
census tract and the metropolitan area.

Three questions are answered.

The first is whether access is worse where deprivation is higher. This is asked
separately for each of the thirteen definitions of a food destination, because the
definitions differ by two orders of magnitude in how far the median resident
has to walk, and a gradient that holds under one definition need not hold under
another. The direction and the size of the gradient are reported for every
definition rather than for a preferred one, so that the answer to the question
can be seen to depend on the definition or to be robust to it.

The second is whether the people who live furthest from a supermarket are the
people least able to drive to one. Walking distance is the binding constraint
for a household with no car and a minor inconvenience for a household with one,
so distance alone does not say who is poorly served. The distribution of
population across distance and vehicle availability together is reported, which
is the structure used in United States federal practice, where a tract is
flagged on distance and the households without a vehicle beyond that distance
are counted separately.

The third is what the measurements look like at the tract and the area, which
is the level at which results are reported and compared.

All quantities are weighted by population. A block holding two thousand people
and a block holding twenty are not equal observations, so an unweighted summary
over blocks describes the geography of blocks rather than the experience of
residents. The population weighted median is the distance of the median
resident.

Two distances are used as thresholds throughout. Five hundred metres is the
catchment conventionally treated as walkable in work on pedestrian access. One
thousand six hundred and nine metres is one mile, the distance behind the
classification of an urban area with no supermarket as a food desert in United
States federal practice.

Deprivation enters through two published measures. The grade of social
backwardness is a five level ordered classification, and the index of urban
marginalisation is continuous. The tables here use the grades, and the fifths
of the index, so that the direction found with one measure can be checked
against the other. Tracts carrying neither are reported and excluded from the
gradient tables.

The index is also carried forward on a standardised scale for the models
estimated later. The published index is standardised over every urban tract in
the country without weights, so within the metropolitan tracts actually
modelled, and weighted by their population, it has neither mean zero nor unit
spread. It is therefore standardised again here, with population weights, over
the estimation sample: the tracts that carry the index, a positive population
and density, and a distance under every one of the thirteen definitions. A
coefficient on the resulting scale is the change per standard deviation of
deprivation among the residents studied. The published scale is kept beside it.
The estimation sample is marked in the output so that every later estimate
rests on the same tracts.

The distance from a block to its network node is summarised by deprivation as
well, since private streets inside gated developments are absent from the
networks, and if their blocks attach to the perimeter the attachment distance
would be longest where deprivation is lowest.

Vehicle availability is the share of dwellings in the tract reporting a car,
and its complement is the share without one.

Inputs are `Results/07_block_access.parquet`, `Results/04_blocks.parquet`,
`Results/04_tracts.parquet` and `Results/06_street_commerce.parquet`.

Outputs are `Results/08_tract_access.parquet` and
`Results/08_area_access.parquet`, carrying the population weighted median
distance and the population shares within each threshold for every destination
set, with the restandardised index and the estimation sample flag on the tract
file, and `Results/08_gradient.parquet`, the gradient of access against
deprivation for every destination set.

Runtime is a few minutes.

## 1. Setup

In [ ]:
import sys, os, importlib

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = os.environ.get('MXO_BASE', '/content/drive/MyDrive/Mexico_Obesity')
os.environ['MXO_BASE'] = PROJECT
sys.path.insert(0, f'{PROJECT}/Code')
import mxo_utils as mx
importlib.reload(mx)

assert mx.BASE == PROJECT, f'the module resolves to {mx.BASE}'

import numpy as np
import pandas as pd

rep = mx.Report('08_access_deprivation').start()
mx.flow_reset()

RESULTS = mx.DIRS['results']
THRESHOLDS = [500, 1609]
WALK_M_PER_MIN = 4.8 * 1000 / 60

## 2. The block panel

Each inhabited block carries its distance to the nearest destination in every
set, its population, and the attributes of the tract it belongs to.

In [ ]:
ACC = pd.read_parquet(os.path.join(RESULTS, '07_block_access.parquet'))
SETS = [c[len('distance_m_'):] for c in ACC.columns
        if c.startswith('distance_m_')]
DCOL = {s: f'distance_m_{s}' for s in SETS}
print(f'{len(ACC):,} inhabited blocks, {len(SETS)} destination sets')
print('sets:', ', '.join(SETS))

# Pairs in which one set contains the other, used where nesting is examined.
PAIRS = [('supermarket_club', 'supermarket_mini'),
         ('fresh_only', 'fresh_abarrotes'),
         ('fresh_only', 'fresh_street'),
         ('all_food', 'all_food_street')]
for a, b in PAIRS:
    assert a in SETS and b in SETS, (a, b)

BLK = pd.read_parquet(os.path.join(RESULTS, '04_blocks.parquet'),
                      columns=['CVEGEO_MZA', 'CVEGEO', 'CVE_ENT',
                               'macro_region'])
TR = pd.read_parquet(os.path.join(RESULTS, '04_tracts.parquet'),
                     columns=['CVEGEO', 'grs_grade', 'grs_grade_ord',
                              'imu_depriv', 'imu_depriv_z', 'veh_share',
                              'tract_complete', 'pop_density_km2'])
ST = pd.read_parquet(os.path.join(RESULTS, '06_street_commerce.parquet'),
                     columns=['CVEGEO_MZA', 'street_any', 'street_known'])

B = (ACC.merge(BLK, on='CVEGEO_MZA', how='left')
        .merge(TR, on='CVEGEO', how='left')
        .merge(ST, on='CVEGEO_MZA', how='left'))
B['carless_share'] = 1 - B['veh_share']

assert B['CVEGEO'].notna().all(), 'a block has no tract'
assert len(B) == len(ACC)

print(f'\n{len(B):,} blocks, {B.POBTOT.sum():,.0f} people, '
      f'{B.CVEGEO.nunique():,} tracts, {B.slug.nunique()} areas')
print(f'blocks whose tract has a deprivation grade    '
      f'{int(B.grs_grade.notna().sum()):,} ({B.grs_grade.notna().mean():.1%})')
print(f'blocks whose tract has a marginalisation score '
      f'{int(B.imu_depriv.notna().sum()):,} ({B.imu_depriv.notna().mean():.1%})')
print(f'blocks whose tract has a vehicle share         '
      f'{int(B.veh_share.notna().sum()):,} ({B.veh_share.notna().mean():.1%})')

mx.flow('inhabited blocks measured', len(B), 'from the routing')
mx.flow('blocks with a deprivation grade', int(B.grs_grade.notna().sum()),
        'tracts without a grade are reported separately')

## 3. Population weighted measures

Three quantities are computed for every group of blocks and every destination
set: the distance of the median resident, and the share of residents within each
of the two thresholds. A block whose distance is missing, because no destination
of that set can reach it through the network, is excluded from the set's own
summary and the number excluded is reported alongside.

In [ ]:
def wmedian(frame, value, weight='POBTOT'):
    """Distance of the median resident."""
    f = frame[[value, weight]].dropna(subset=[value])
    if not len(f) or f[weight].sum() <= 0:
        return np.nan
    f = f.sort_values(value)
    cw = f[weight].cumsum() / f[weight].sum()
    return float(f.loc[cw >= 0.5, value].iloc[0])


def group_wmedian(frame, by, value, weight='POBTOT'):
    """Distance of the median resident within each group, vectorised."""
    f = frame[[by, value, weight]].dropna(subset=[value])
    f = f[f[weight] > 0].sort_values([by, value])
    total = f.groupby(by, sort=False)[weight].transform('sum')
    cumulative = f.groupby(by, sort=False)[weight].cumsum()
    first = (cumulative >= 0.5 * total)
    return f[first].groupby(by, sort=False)[value].first()


def group_profile(frame, by, weight='POBTOT'):
    """Median distance and threshold shares per group, for every set."""
    out = frame.groupby(by, sort=False).agg(
        blocks=('CVEGEO_MZA', 'size'), population=(weight, 'sum'))
    for s in SETS:
        col = DCOL[s]
        ok = frame[col].notna()
        w = frame[weight].where(ok, 0.0)
        out[f'pop_measured_{s}'] = w.groupby(frame[by]).sum()
        out[f'median_m_{s}'] = group_wmedian(frame, by, col, weight)
        for t in THRESHOLDS:
            inside = frame[weight].where(ok & (frame[col] <= t), 0.0)
            out[f'within{t}_{s}'] = (inside.groupby(frame[by]).sum()
                                     / out[f'pop_measured_{s}']).round(4)
    return out

## 4. Access against the grade of social backwardness

The grade is an ordered classification running from very low to very high
backwardness, so a gradient appears as a monotone change across its five
levels. The table gives, for every destination set, the distance of the median
resident at each grade.

The ratio between the most and the least deprived grade states the direction
and the size of the gradient in one number. A ratio above one means residents
of the most deprived tracts walk further than residents of the least deprived
ones. A ratio below one means they walk less far.

In [ ]:
GRADED = B[B.grs_grade.notna()].copy()
print(f'{len(GRADED):,} blocks, {GRADED.POBTOT.sum():,.0f} people, in tracts '
      f'with a grade')
print(f'{len(B) - len(GRADED):,} blocks, '
      f'{B.POBTOT.sum() - GRADED.POBTOT.sum():,.0f} people, in tracts without '
      f'one, excluded from this section\n')

rows = []
for s in SETS:
    r = {'destination set': s}
    for g in mx.GRADE_ORDER:
        r[g] = wmedian(GRADED[GRADED.grs_grade == g], DCOL[s])
    rows.append(r)
GRAD = pd.DataFrame(rows).set_index('destination set')
print('--- distance of the median resident, metres, by grade ---\n')
print(GRAD.round(0).to_string())

# A set whose median is zero at the least deprived grade has no ratio to
# report, because the median resident there is already at a destination, so the
# direction is read from the difference instead.
low, high = mx.GRADE_ORDER[0], mx.GRADE_ORDER[-1]
GRAD['ratio most to least deprived'] = np.where(
    GRAD[low] > 0, (GRAD[high] / GRAD[low]).round(2), np.nan)
GRAD['difference, most less least'] = (GRAD[high] - GRAD[low]).round(0)
GRAD['direction'] = np.where(
    GRAD[high] > GRAD[low], 'deprived walk further',
    np.where(GRAD[high] < GRAD[low], 'deprived walk less far', 'no difference'))
print('\n--- direction and size of the gradient ---\n')
print(GRAD[[low, high, 'difference, most less least',
            'ratio most to least deprived', 'direction']].to_string())
rep.add_table('median distance by grade of social backwardness',
              GRAD.reset_index(), max_rows=12)

In [ ]:
print('--- share of residents within 500 metres, by grade ---\n')
rows = []
for s in SETS:
    r = {'destination set': s}
    for g in mx.GRADE_ORDER:
        sub = GRADED[GRADED.grs_grade == g]
        ok = sub[DCOL[s]].notna()
        pop = sub.POBTOT.where(ok, 0.0).sum()
        r[g] = round(float(sub.POBTOT.where(
            ok & (sub[DCOL[s]] <= 500), 0.0).sum() / pop), 4) if pop else np.nan
    r['difference, most less least'] = round(
        r[mx.GRADE_ORDER[-1]] - r[mx.GRADE_ORDER[0]], 4)
    rows.append(r)
W500 = pd.DataFrame(rows).set_index('destination set')
print(W500.to_string())
rep.add_table('share within 500 metres by grade', W500.reset_index(),
              max_rows=12)

print('\n--- distance from a block to its network node, by grade ---\n')


def wquantile(frame, value, q, weight='POBTOT'):
    f = frame[[value, weight]].dropna().sort_values(value)
    cw = f[weight].cumsum() / f[weight].sum()
    return float(f.loc[cw >= q, value].iloc[0])


rows = []
for g in mx.GRADE_ORDER:
    sub = GRADED[GRADED.grs_grade == g]
    rows.append({'grade': g, 'blocks': len(sub),
                 'median snap m': round(wmedian(sub, 'snap_m'), 1),
                 'p90 snap m': round(wquantile(sub, 'snap_m', 0.9), 1),
                 'share beyond 50 m': round(float(
                     sub.POBTOT.where(sub.snap_m > 50, 0.0).sum()
                     / sub.POBTOT.sum()), 4)})
SNAP = pd.DataFrame(rows)
print(SNAP.to_string(index=False))
rep.add_table('block attachment distance by grade', SNAP, max_rows=10)

print('\n--- population and blocks behind each grade ---\n')
print(GRADED.groupby('grs_grade')
            .agg(blocks=('CVEGEO_MZA', 'size'), population=('POBTOT', 'sum'),
                 median_carless=('carless_share', 'median'),
                 median_density=('pop_density_km2', 'median'))
            .reindex(mx.GRADE_ORDER).round(3).to_string())

## 5. Access against the index of urban marginalisation

The index is continuous, so residents are divided into five groups of equal
population by the value of the index for their tract. Equal groups of
population rather than of tracts keep the comparison with the grades on the
same footing, since each group then describes a fifth of the people rather than
a fifth of the places.

In [ ]:
def population_quintiles(frame, col, weight='POBTOT', k=5, labels=None):
    """Groups of equal population, ordered by col."""
    f = frame[[col, weight]].dropna(subset=[col]).sort_values(col)
    cw = f[weight].cumsum() / f[weight].sum()
    cuts = sorted({f.loc[cw >= q / k, col].iloc[0] for q in range(1, k)})
    edges = [-np.inf] + cuts + [np.inf]
    if len(edges) - 1 != len(labels):
        raise RuntimeError(
            f'{col} takes too few distinct values to form {k} groups of equal '
            f'population: boundaries {cuts}')
    return pd.cut(frame[col], bins=edges, labels=labels), cuts


IMU_LAB = ['least deprived fifth', 'second', 'third', 'fourth',
           'most deprived fifth']
B['imu_group'], imu_cuts = population_quintiles(B, 'imu_depriv',
                                                labels=IMU_LAB)
SCORED = B[B.imu_group.notna()].copy()
print(f'{len(SCORED):,} blocks, {SCORED.POBTOT.sum():,.0f} people, in tracts '
      f'with a score')
print('index values at the group boundaries:',
      [round(x, 4) for x in imu_cuts], '\n')

rows = []
for s in SETS:
    r = {'destination set': s}
    for g in IMU_LAB:
        r[g] = wmedian(SCORED[SCORED.imu_group == g], DCOL[s])
    r['difference, most less least'] = round(r[IMU_LAB[-1]] - r[IMU_LAB[0]])
    r['ratio most to least'] = (round(r[IMU_LAB[-1]] / r[IMU_LAB[0]], 2)
                                if r[IMU_LAB[0]] else np.nan)
    rows.append(r)
IMU = pd.DataFrame(rows).set_index('destination set')
print('--- distance of the median resident, metres, by fifth of population ---\n')
print(IMU.round(0).to_string())
rep.add_table('median distance by fifth of marginalisation', IMU.reset_index(),
              max_rows=12)

print('\n--- the two measures compared on direction ---\n')
cmp = pd.DataFrame({
    'grade ratio': GRAD['ratio most to least deprived'],
    'index ratio': IMU['ratio most to least']})
cmp['grade difference'] = GRAD['difference, most less least']
cmp['index difference'] = IMU['difference, most less least']
cmp['same direction'] = ((cmp['grade difference'] > 0) ==
                         (cmp['index difference'] > 0))
print(cmp.to_string())
rep.add_table('gradient direction under both deprivation measures',
              cmp.reset_index(), max_rows=12)

## 6. Distance and vehicle availability together

Walking distance constrains a household with no car and inconveniences a
household with one, so the two have to be read together. Residents are divided
into five groups of equal population by the share of dwellings in their tract
without a car, and the distribution of population across those groups and
across distance is reported.

The headline quantity is the population living beyond one mile from a
supermarket in a tract where most dwellings have no car, which is the
combination that United States federal practice treats as the case of poor
access.

In [ ]:
CAR_LAB = ['fewest without a car', 'second', 'third', 'fourth',
           'most without a car']
B['carless_group'], car_cuts = population_quintiles(B, 'carless_share',
                                                    labels=CAR_LAB)
CAR = B[B.carless_group.notna() & B['distance_m_supermarket_club'].notna()].copy()
print(f'{len(CAR):,} blocks, {CAR.POBTOT.sum():,.0f} people, with both a '
      f'vehicle share and a distance to a supermarket')
print('share of dwellings without a car at the group boundaries:',
      [round(x, 3) for x in car_cuts], '\n')

print('--- distance to a supermarket, by how many households have no car ---\n')
rows = []
for g in CAR_LAB:
    sub = CAR[CAR.carless_group == g]
    r = {'group': g, 'population': int(sub.POBTOT.sum()),
         'median carless share': round(float(sub.carless_share.median()), 3),
         'median metres': round(wmedian(sub, 'distance_m_supermarket_club'))}
    for t in THRESHOLDS:
        r[f'share beyond {t} m'] = round(float(
            sub.POBTOT.where(sub['distance_m_supermarket_club'] > t, 0.0).sum()
            / sub.POBTOT.sum()), 4)
    rows.append(r)
CARTAB = pd.DataFrame(rows)
print(CARTAB.to_string(index=False))
rep.add_table('distance to a supermarket by vehicle availability', CARTAB,
              max_rows=10)

In [ ]:
BANDS = [0, 500, 1609, np.inf]
NAMES = ['within 500 m', '500 m to one mile', 'beyond one mile']
CAR['band'] = pd.cut(CAR['distance_m_supermarket_club'], bins=BANDS,
                     labels=NAMES, include_lowest=True)

grid = (CAR.pivot_table(index='carless_group', columns='band',
                        values='POBTOT', aggfunc='sum', observed=False)
           .reindex(CAR_LAB)[NAMES])
print('--- population by distance to a supermarket and vehicle availability ---\n')
print(grid.fillna(0).astype('int64').to_string())
print('\nas a share of all residents measured:\n')
print((grid / grid.values.sum()).round(4).to_string())
rep.add_table('population by distance band and vehicle availability',
              grid.fillna(0).astype('int64').reset_index(), max_rows=10)

far = CAR['distance_m_supermarket_club'] > 1609
majority_carless = CAR['carless_share'] > 0.5
both = far & majority_carless
print(f'\nbeyond one mile from a supermarket            '
      f'{int(CAR.loc[far, "POBTOT"].sum()):,} '
      f'({CAR.loc[far, "POBTOT"].sum() / CAR.POBTOT.sum():.2%})')
print(f'in a tract where most dwellings have no car  '
      f'{int(CAR.loc[majority_carless, "POBTOT"].sum()):,} '
      f'({CAR.loc[majority_carless, "POBTOT"].sum() / CAR.POBTOT.sum():.2%})')
print(f'both                                         '
      f'{int(CAR.loc[both, "POBTOT"].sum()):,} '
      f'({CAR.loc[both, "POBTOT"].sum() / CAR.POBTOT.sum():.2%})')

print('\n--- the doubly constrained population, by grade and by area ---\n')
print(CAR.loc[both].groupby('grs_grade', observed=False)
         .agg(blocks=('CVEGEO_MZA', 'size'), population=('POBTOT', 'sum'))
         .reindex(mx.GRADE_ORDER).to_string())
print()
top = (CAR.assign(pop_both=CAR.POBTOT.where(both, 0.0))
          .groupby('slug').agg(population=('POBTOT', 'sum'),
                               pop_both=('pop_both', 'sum')))
top['share'] = (top.pop_both / top.population).round(4)
print(top.nlargest(15, 'share').to_string())
rep.add_table('doubly constrained population by area', top.reset_index(),
              max_rows=92)

## 7. The tract and the area

The block level measurements are carried up to the two levels at which results
are reported. Each tract and each area receives the distance of its median
resident and the share of its residents within each threshold, for every
destination set.

Each set is summarised over the residents it reaches. A block that no member of
a set can reach through the network carries no distance for that set, so it is
absent from that set's median and from the denominator of its threshold shares,
while still counting towards a wider set that does reach it. The population
each set was measured over is therefore recorded beside the measure. One
consequence is that in a small number of tracts the median for a wider set
exceeds the median for a narrower one, because the two were taken over
different residents, and the number of tracts in which that happens is
reported.

In [ ]:
TRACT = group_profile(B, 'CVEGEO')
TRACT = TRACT.join(TR.set_index('CVEGEO')[
    ['grs_grade', 'grs_grade_ord', 'imu_depriv', 'imu_depriv_z', 'veh_share',
     'tract_complete', 'pop_density_km2']])
TRACT = TRACT.join(B.groupby('CVEGEO')[['slug', 'macro_region']].first())
print(f'{len(TRACT):,} tracts')

# The estimation sample: tracts carrying the index, a positive population and
# density, and a distance under every definition. Every model estimated later
# is fitted on these tracts, and the index is standardised over them with
# population weights so that a coefficient is the change per standard
# deviation of deprivation among the residents studied.
MED = [f'median_m_{s}' for s in SETS]
TRACT['estimation_sample'] = (TRACT['imu_depriv'].notna()
                              & (TRACT['population'] > 0)
                              & (TRACT['pop_density_km2'] > 0)
                              & TRACT[MED].notna().all(axis=1))
E = TRACT[TRACT.estimation_sample]
w_e = E['population'].to_numpy(float)
m_e = float((w_e * E['imu_depriv']).sum() / w_e.sum())
s_e = float(np.sqrt((w_e * (E['imu_depriv'] - m_e) ** 2).sum() / w_e.sum()))
TRACT = TRACT.rename(columns={'imu_depriv_z': 'imu_depriv_z_national'})
TRACT['imu_depriv_z'] = (TRACT['imu_depriv'] - m_e) / s_e

print(f'\nestimation sample: {len(E):,} tracts, {w_e.sum():,.0f} people, '
      f'{E.slug.nunique()} areas')
reasons = pd.DataFrame([
    {'condition': 'index present', 'tracts failing': int(TRACT.imu_depriv.isna().sum())},
    {'condition': 'positive population', 'tracts failing': int((TRACT.population <= 0).sum())},
    {'condition': 'positive density',
     'tracts failing': int((~(TRACT.pop_density_km2 > 0)).sum())},
    {'condition': 'distance under every definition',
     'tracts failing': int((~TRACT[MED].notna().all(axis=1)).sum())}])
print(reasons.to_string(index=False))
rep.add_table('tracts outside the estimation sample, by condition', reasons,
              max_rows=10)

def wstats(x, w):
    m = float((w * x).sum() / w.sum())
    return m, float(np.sqrt((w * (x - m) ** 2).sum() / w.sum()))

print('\n--- the index within the estimation sample ---\n')
rows = []
for col, label in [('imu_depriv_z_national', 'published scale, standardised '
                                             'over all urban tracts unweighted'),
                   ('imu_depriv_z', 'restandardised here, population weighted')]:
    m, s = wstats(TRACT.loc[E.index, col].to_numpy(float), w_e)
    um, us = float(TRACT.loc[E.index, col].mean()), float(TRACT.loc[E.index, col].std())
    rows.append({'scale': label, 'weighted mean': round(m, 4),
                 'weighted sd': round(s, 4), 'unweighted mean': round(um, 4),
                 'unweighted sd': round(us, 4)})
SCALE = pd.DataFrame(rows)
print(SCALE.to_string(index=False))
s_n = float(TRACT.imu_depriv.std() / TRACT.imu_depriv_z_national.std())
print(f'\none standard deviation on the new scale is {s_e / s_n:.4f} standard '
      f'deviations on the published scale, so a coefficient per unit of the '
      f'new scale is {s_e / s_n:.4f} times the coefficient per published unit')
rep.add_table('the marginalisation index on both scales', SCALE, max_rows=10)

AREA = group_profile(B, 'slug')
metros = mx.load_metros()
tipo = metros.assign(slug=mx._slug_name(metros['metro_key']).values).set_index(
    'slug')[['metro_name', 'tipo', 'tier', 'conapo_pop_2020']]
AREA = AREA.join(tipo)
assert AREA['tipo'].notna().all()
print(f'{len(AREA)} areas\n')

show = ['metro_name', 'tipo', 'population'] + \
       [f'median_m_{s}' for s in ['supermarket_club', 'fresh_only',
                                  'fresh_street', 'all_food_street']]
print('--- the fifteen areas furthest from a supermarket ---\n')
print(AREA.nlargest(15, 'median_m_supermarket_club')[show].to_string())
print('\n--- the fifteen closest ---\n')
print(AREA.nsmallest(15, 'median_m_supermarket_club')[show].to_string())
rep.add_table('access by area', AREA.reset_index(), max_rows=92)

In [ ]:
print('--- by official classification ---\n')
rows = []
for t in mx.TIPO_ORDER:
    sub = B[B.slug.isin(AREA.index[AREA.tipo == t])]
    r = {'tipo': t, 'areas': int((AREA.tipo == t).sum()),
         'population': int(sub.POBTOT.sum())}
    r.update({s: round(wmedian(sub, DCOL[s])) for s in SETS})
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))
rep.add_table('median distance by official classification',
              pd.DataFrame(rows), max_rows=10)

print('\n--- by macro region ---\n')
rows = []
for g, sub in B.groupby('macro_region'):
    r = {'macro_region': g, 'population': int(sub.POBTOT.sum()),
         'median carless share': round(float(sub.carless_share.median()), 3)}
    r.update({s: round(wmedian(sub, DCOL[s])) for s in SETS})
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))
rep.add_table('median distance by macro region', pd.DataFrame(rows),
              max_rows=10)

## 8. Checks

Each check names the condition under which it fails.

In [ ]:
print('--- checks ---\n')

d = int(B['CVEGEO_MZA'].duplicated().sum())
print(f'1. duplicate blocks                          {d}')
assert d == 0

print(f'2. population, panel against routing         {B.POBTOT.sum():,.0f}')

# A tract or an area losing population in the aggregation would understate it.
gap_t = abs(TRACT.population.sum() - B.POBTOT.sum())
gap_a = abs(AREA.population.sum() - B.POBTOT.sum())
print(f'3. population, tracts against blocks         {gap_t:,.0f}')
print(f'4. population, areas against blocks          {gap_a:,.0f}')
assert gap_t < 1 and gap_a < 1

# Nesting holds block by block. After aggregation it holds wherever the two
# sets are measured on the same residents, so that is what is asserted. Where
# they are not, the medians can cross, and the number of groups in which they
# do is reported beside it.
rows = []
for narrow, wide in PAIRS:
    both_present = B[B[DCOL[narrow]].notna() & B[DCOL[wide]].notna()]
    for level, by, frame in [('tract', 'CVEGEO', TRACT), ('area', 'slug', AREA)]:
        same = pd.concat(
            [group_wmedian(both_present, by, DCOL[narrow]).rename('n'),
             group_wmedian(both_present, by, DCOL[wide]).rename('w')],
            axis=1).dropna()
        raw = pd.concat([frame[f'median_m_{narrow}'].rename('n'),
                         frame[f'median_m_{wide}'].rename('w')],
                        axis=1).dropna()
        rows.append({'level': level, 'narrower': narrow, 'wider': wide,
                     'violating on the same residents':
                         int((same.w > same.n + 1e-6).sum()),
                     'medians crossing': int((raw.w > raw.n + 1e-6).sum())})
V = pd.DataFrame(rows)
print('5. nesting after aggregation')
print(V.to_string(index=False))
assert V['violating on the same residents'].sum() == 0

crossing = set()
for narrow, wide in PAIRS:
    raw = TRACT[[f'median_m_{narrow}', f'median_m_{wide}']].dropna()
    crossing |= set(raw.index[raw[f'median_m_{wide}']
                              > raw[f'median_m_{narrow}'] + 1e-6])
pop_crossing = float(TRACT.loc[sorted(crossing), 'population'].sum())     if crossing else 0.0
print(f'   tracts where a pair of medians cross      {len(crossing):,}, '
      f'holding {pop_crossing:,.0f} people '
      f'({pop_crossing / TRACT.population.sum():.3%})')

# Every group of equal population must hold about a fifth of the residents.
for label, col, labs in [('marginalisation', 'imu_group', IMU_LAB),
                         ('vehicle availability', 'carless_group', CAR_LAB)]:
    sh = (B.groupby(col, observed=False)['POBTOT'].sum()
            / B.loc[B[col].notna(), 'POBTOT'].sum()).reindex(labs).round(3)
    print(f'6. shares of population across the {label} groups')
    print('   ', sh.to_dict())
    assert sh.between(0.1, 0.3).all(), sh.to_dict()

# A share cannot fall outside the unit interval.
bad = 0
for s in SETS:
    for t in THRESHOLDS:
        v = TRACT[f'within{t}_{s}'].dropna()
        bad += int(((v < 0) | (v > 1)).sum())
print(f'7. threshold shares outside zero to one     {bad}')
assert bad == 0

# The restandardised index has weighted mean zero and unit weighted spread
# within the estimation sample.
m_chk, s_chk = wstats(TRACT.loc[TRACT.estimation_sample, 'imu_depriv_z']
                      .to_numpy(float), w_e)
print(f'   index within the estimation sample, weighted mean {m_chk:.2e}, '
      f'weighted sd {s_chk:.6f}')
assert abs(m_chk) < 1e-8 and abs(s_chk - 1) < 1e-8
print(f'   tracts in the estimation sample           '
      f'{int(TRACT.estimation_sample.sum()):,}')

print(f'\n8. tracts                                    {len(TRACT):,}')
print(f'9. areas                                     {len(AREA)}')
print(f'10. gradient directions, supermarkets        '
      f'{GRAD.loc["supermarket_club", "direction"]}')
print(f'11. gradient directions, fresh and staple    '
      f'{GRAD.loc["fresh_only", "direction"]}')
print(f'12. definitions whose gradient runs the other way '
      f'{int((GRAD["difference, most less least"] < 0).sum())} of {len(SETS)}')

## 9. Write results

In [ ]:
outt = os.path.join(RESULTS, '08_tract_access.parquet')
TRACT.reset_index().to_parquet(outt, index=False)
print('written:', outt, f'({os.path.getsize(outt) / 1e6:.1f} MB)')

outa = os.path.join(RESULTS, '08_area_access.parquet')
AREA.reset_index().to_parquet(outa, index=False)
print('written:', outa, f'({os.path.getsize(outa) / 1e6:.1f} MB)')

G = GRAD.reset_index().assign(measure='grade of social backwardness')
I = IMU.reset_index().assign(measure='index of urban marginalisation')
outg = os.path.join(RESULTS, '08_gradient.parquet')
pd.concat([G, I], ignore_index=True).to_parquet(outg, index=False)
print('written:', outg, f'({os.path.getsize(outg) / 1e6:.1f} MB)')

mx.flow('blocks in the panel', len(B), 'from the routing')
mx.flow('blocks with a deprivation grade', len(GRADED),
        'used for the gradient against the grades')
mx.flow('blocks with a marginalisation score', len(SCORED),
        'used for the gradient against the index')
mx.flow('blocks with a vehicle share and a supermarket distance', len(CAR),
        'used for the distance and vehicle table')

print()
print(mx.flow_table().to_string(index=False))
mx.save_flow(os.path.join(mx.DIRS['logs'], '08_sample_flow.csv'))
rep.add_table('attrition', mx.flow_table(), max_rows=20)

flipped = GRAD.index[GRAD['difference, most less least'] < 0].tolist()
rep.note(f'{len(TRACT):,} tracts and {len(AREA)} areas, '
         f'{int(TRACT.estimation_sample.sum()):,} tracts in the estimation '
         f'sample; the gradient against '
         f'deprivation runs the other way under {len(flipped)} of {len(SETS)} '
         f'definitions ({", ".join(flipped) if flipped else "none"}); '
         f'{CAR.loc[both, "POBTOT"].sum() / CAR.POBTOT.sum():.2%} of residents '
         f'live beyond one mile from a supermarket in a tract where most '
         f'dwellings have no car')
rep.save()